# Stereo Vision Self-Learning Notebook

This notebook is a guided, hands-on chapter for learning **stereo vision** in computer vision.
It follows a concept-first, code-second format: each section introduces the key idea, then gives a small experiment or task.

## Learning goals

By the end of this notebook, you should be able to:

1. Explain how two calibrated cameras can reconstruct 3D points.
2. Derive the relationship between disparity and depth.
3. Explain why stereo rectification makes correspondence search easier.
4. Implement simple stereo block matching with SSD and NCC costs.
5. Diagnose common stereo failure cases: textureless regions, repeated patterns, specularities, and depth discontinuities.
6. Understand how smoothness terms and structured light can improve matching.

## How to use this notebook

Run the cells from top to bottom. For each task, first predict what will happen, then run the code and compare your observation with the prediction.

## 0. Setup

We will use only standard scientific Python tools. The images are synthetic so that the notebook can run anywhere without downloading data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(7)


def normalize01(x):
    """Normalize an array to the range [0, 1]."""
    x = np.asarray(x, dtype=np.float32)
    lo, hi = np.nanmin(x), np.nanmax(x)
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def show_image(img, title=None, cmap="gray", vmin=None, vmax=None):
    """Display one image with a compact layout."""
    plt.figure(figsize=(5, 4))
    plt.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


def show_grid(images, titles=None, cmap="gray", vmin=None, vmax=None, figsize=(14, 4)):
    """Display a row of images."""
    n = len(images)
    titles = titles or [None] * n
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles), start=1):
        plt.subplot(1, n, i)
        plt.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
        if title:
            plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 1. Stereo geometry and disparity

### Concept

A stereo rig uses two images of the same 3D scene. In the simplest calibrated and rectified setup, the two cameras are parallel and separated by a horizontal baseline $B$.

For a pinhole camera, projection can be written as:

$$
s
\begin{bmatrix}
u \\
v \\
1
\end{bmatrix}
=
K
\begin{bmatrix}
R & t
\end{bmatrix}
\begin{bmatrix}
X \\
Y \\
Z \\
1
\end{bmatrix}
$$

For parallel stereo cameras with focal length $f$ and baseline $B$:

$$
u_L = \frac{fX}{Z} + c_x
$$

$$
u_R = \frac{f(X-B)}{Z} + c_x
$$

The horizontal disparity is:

$$
d = u_L - u_R
$$

Substituting the two projection equations gives:

$$
d = \frac{fB}{Z}
$$

Therefore depth is inversely proportional to disparity:

$$
Z = \frac{fB}{d}
$$

### Task 1

Before running the next cell, predict what happens to disparity when:

- The object gets farther from the cameras.
- The baseline $B$ increases.
- The focal length $f$ increases.

In [ ]:
# Parallel stereo parameters
f = 800.0      # focal length in pixels
B = 0.12       # baseline in meters
cx, cy = 320.0, 240.0

# Some 3D points in front of the cameras
Z = np.array([0.6, 1.0, 1.5, 2.5, 4.0])
X = np.array([0.00, 0.05, -0.08, 0.12, -0.10])
Y = np.zeros_like(X)

u_left = f * X / Z + cx
u_right = f * (X - B) / Z + cx
disparity = u_left - u_right
Z_recovered = f * B / disparity

print("Depth Z in meters:       ", Z)
print("Disparity in pixels:     ", disparity)
print("Recovered depth Z:       ", Z_recovered)
print("Maximum abs depth error: ", np.max(np.abs(Z - Z_recovered)))

plt.figure(figsize=(6, 4))
plt.plot(Z, disparity, marker="o")
plt.xlabel("Depth Z (meters)")
plt.ylabel("Disparity d (pixels)")
plt.title("Disparity decreases as depth increases")
plt.grid(True)
plt.show()

### Mini-exercise

Change `B` from `0.12` to `0.24`. Then change `f` from `800` to `1200`.

Write down what happens to the disparity curve. Explain your answer using:

$$
d = \frac{fB}{Z}
$$

## 2. Triangulation

### Concept

If we know the projection matrices of two cameras and the matching image points, we can recover a 3D point by triangulation.

Let a 3D point in homogeneous coordinates be $\mathbf{X}$, and let its image point be $\mathbf{x}$. The projection equation is:

$$
\mathbf{x} \sim P\mathbf{X}
$$

The cross-product constraint is:

$$
\mathbf{x} \times (P\mathbf{X}) = \mathbf{0}
$$

This gives a linear system:

$$
A\mathbf{X} = \mathbf{0}
$$

We solve it with SVD and convert the homogeneous result back to Euclidean coordinates.

### Task 2

Run the triangulation code. Then add small Gaussian noise to the image points and observe how the 3D reconstruction error changes.

In [ ]:
def make_K(f=800.0, cx=320.0, cy=240.0):
    """Create a simple camera intrinsic matrix."""
    return np.array([[f, 0.0, cx],
                     [0.0, f, cy],
                     [0.0, 0.0, 1.0]], dtype=np.float64)


def project(P, X_world):
    """Project 3D points with a 3x4 camera matrix."""
    X_h = np.c_[X_world, np.ones(len(X_world))]
    x_h = (P @ X_h.T).T
    return x_h[:, :2] / x_h[:, 2:3]


def triangulate_dlt(P1, P2, x1, x2):
    """Triangulate one point from two image observations using DLT."""
    u1, v1 = x1
    u2, v2 = x2
    A = np.array([
        u1 * P1[2] - P1[0],
        v1 * P1[2] - P1[1],
        u2 * P2[2] - P2[0],
        v2 * P2[2] - P2[1],
    ])
    _, _, Vt = np.linalg.svd(A)
    X_h = Vt[-1]
    return X_h[:3] / X_h[3]

# Camera matrices for parallel stereo
K = make_K(f=f, cx=cx, cy=cy)
R_left = np.eye(3)
t_left = np.zeros((3, 1))
R_right = np.eye(3)
t_right = np.array([[-B], [0.0], [0.0]])

P_left = K @ np.hstack([R_left, t_left])
P_right = K @ np.hstack([R_right, t_right])

# Create synthetic 3D points
X_world = np.array([
    [-0.10, -0.05, 1.00],
    [ 0.05,  0.03, 1.40],
    [ 0.15, -0.02, 2.00],
    [-0.20,  0.08, 2.80],
])

x_left = project(P_left, X_world)
x_right = project(P_right, X_world)

X_hat = np.array([
    triangulate_dlt(P_left, P_right, xl, xr)
    for xl, xr in zip(x_left, x_right)
])

print("Original 3D points:")
print(X_world)
print("\nTriangulated 3D points:")
print(X_hat)
print("\nMean reconstruction error:", np.linalg.norm(X_world - X_hat, axis=1).mean())

In [ ]:
# Add image noise and observe the reconstruction error.
noise_std_pixels = 0.5
x_left_noisy = x_left + rng.normal(0, noise_std_pixels, size=x_left.shape)
x_right_noisy = x_right + rng.normal(0, noise_std_pixels, size=x_right.shape)

X_hat_noisy = np.array([
    triangulate_dlt(P_left, P_right, xl, xr)
    for xl, xr in zip(x_left_noisy, x_right_noisy)
])

errors = np.linalg.norm(X_world - X_hat_noisy, axis=1)
print("Reconstruction errors with noisy image points:")
print(errors)
print("Mean error:", errors.mean())

# Student task: try 0.1, 1.0, and 2.0 pixels of noise.

## 3. Why rectification matters

### Concept

In a general two-view setup, the matching point in the right image lies somewhere on an epipolar line. This is expressed by the epipolar constraint:

$$
\mathbf{x}_R^T F\mathbf{x}_L = 0
$$

Searching along arbitrary epipolar lines is expensive and error-prone. Stereo rectification warps both images so that corresponding epipolar lines become horizontal scanlines.

Rectification applies one homography to each image:

$$
\tilde{\mathbf{x}}_L = H_L\mathbf{x}_L
$$

$$
\tilde{\mathbf{x}}_R = H_R\mathbf{x}_R
$$

After rectification, corresponding points should satisfy:

$$
\tilde{v}_L \approx \tilde{v}_R
$$

This reduces correspondence search from a 2D problem to a 1D scanline problem.

### Task 3

The next code cell compares a non-parallel camera pair with a parallel rectified-style pair. Look at the vertical coordinate differences. Smaller vertical differences mean easier scanline matching.

In [ ]:
def rot_y(theta_degrees):
    """Rotation around the y-axis."""
    t = np.deg2rad(theta_degrees)
    return np.array([[ np.cos(t), 0.0, np.sin(t)],
                     [ 0.0,       1.0, 0.0],
                     [-np.sin(t), 0.0, np.cos(t)]])


def rot_z(theta_degrees):
    """Rotation around the z-axis."""
    t = np.deg2rad(theta_degrees)
    return np.array([[np.cos(t), -np.sin(t), 0.0],
                     [np.sin(t),  np.cos(t), 0.0],
                     [0.0,        0.0,       1.0]])

# Build a grid of 3D points.
grid_x, grid_y = np.meshgrid(np.linspace(-0.35, 0.35, 8), np.linspace(-0.2, 0.2, 5))
grid_z = 1.5 + 0.7 * rng.random(grid_x.shape)
points = np.c_[grid_x.ravel(), grid_y.ravel(), grid_z.ravel()]

# Left camera stays fixed.
P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])

# Right camera with a small rotation: this creates non-horizontal epipolar geometry.
C_right = np.array([[B], [0.0], [0.0]])
R_unrectified = rot_z(2.0) @ rot_y(-5.0)
t_unrectified = -R_unrectified @ C_right
P2_unrectified = K @ np.hstack([R_unrectified, t_unrectified])

# Parallel right camera: this is the ideal rectified geometry.
P2_parallel = K @ np.hstack([np.eye(3), np.array([[-B], [0.0], [0.0]])])

x1 = project(P1, points)
x2_unrectified = project(P2_unrectified, points)
x2_parallel = project(P2_parallel, points)

vertical_diff_unrectified = x1[:, 1] - x2_unrectified[:, 1]
vertical_diff_parallel = x1[:, 1] - x2_parallel[:, 1]

print("Mean |vertical difference| before rectified-style alignment:", np.mean(np.abs(vertical_diff_unrectified)))
print("Mean |vertical difference| in ideal parallel stereo:        ", np.mean(np.abs(vertical_diff_parallel)))

plt.figure(figsize=(7, 5))
plt.scatter(x1[:, 0], x1[:, 1], label="Left image points")
plt.scatter(x2_unrectified[:, 0], x2_unrectified[:, 1], label="Right points, non-parallel")
for a, b in zip(x1, x2_unrectified):
    plt.plot([a[0], b[0]], [a[1], b[1]], linewidth=0.7)
plt.gca().invert_yaxis()
plt.xlabel("u")
plt.ylabel("v")
plt.title("Non-parallel cameras: matches are not on the same scanline")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 5))
plt.scatter(x1[:, 0], x1[:, 1], label="Left image points")
plt.scatter(x2_parallel[:, 0], x2_parallel[:, 1], label="Right points, ideal parallel")
for a, b in zip(x1, x2_parallel):
    plt.plot([a[0], b[0]], [a[1], b[1]], linewidth=0.7)
plt.gca().invert_yaxis()
plt.xlabel("u")
plt.ylabel("v")
plt.title("Ideal rectified geometry: matches lie on horizontal scanlines")
plt.legend()
plt.grid(True)
plt.show()

## 4. Matching costs: SAD, SSD, and NCC

### Concept

Once images are rectified, we can compare a patch in the left image with candidate patches along the same row in the right image.

For two patches $p$ and $q$, common costs are:

$$
\operatorname{SAD}(p,q)=\sum_i |p_i-q_i|
$$

$$
\operatorname{SSD}(p,q)=\sum_i (p_i-q_i)^2
$$

Normalized cross-correlation is a similarity score rather than a distance:

$$
\operatorname{NCC}(p,q)=
\frac{\sum_i (p_i-\bar{p})(q_i-\bar{q})}
{\sqrt{\sum_i (p_i-\bar{p})^2}\sqrt{\sum_i (q_i-\bar{q})^2}+\epsilon}
$$

SAD and SSD are sensitive to brightness changes. NCC is more robust to brightness and contrast changes because it subtracts the mean and divides by the standard deviation.

### Task 4

Look at the costs for four candidate patches. Which method best identifies the same pattern when brightness and contrast change?

In [ ]:
def sad(p, q):
    """Sum of absolute differences."""
    return np.sum(np.abs(p - q))


def ssd(p, q):
    """Sum of squared differences."""
    return np.sum((p - q) ** 2)


def ncc(p, q, eps=1e-8):
    """Normalized cross-correlation."""
    p0 = p - np.mean(p)
    q0 = q - np.mean(q)
    return np.sum(p0 * q0) / (np.sqrt(np.sum(p0 ** 2)) * np.sqrt(np.sum(q0 ** 2)) + eps)

patch = np.array([0.1, 0.2, 0.9, 0.8, 0.2, 0.1])
candidates = {
    "same patch": patch.copy(),
    "brighter patch": patch + 0.25,
    "higher contrast patch": 1.8 * (patch - patch.mean()) + patch.mean(),
    "wrong pattern": np.array([0.9, 0.8, 0.1, 0.2, 0.8, 0.9]),
}

print(f"{'Candidate':<24} {'SAD':>10} {'SSD':>10} {'NCC':>10}")
print("-" * 58)
for name, cand in candidates.items():
    print(f"{name:<24} {sad(patch, cand):10.3f} {ssd(patch, cand):10.3f} {ncc(patch, cand):10.3f}")

## 5. Stereo block matching

### Concept

For each pixel $(x,y)$ in the left image, block matching searches possible disparities $d$ and chooses the candidate with the best matching cost.

Using SSD, the matching cost is:

$$
C(x,y,d)=\sum_{(i,j)\in W}
\left(I_L(x+i,y+j)-I_R(x-d+i,y+j)\right)^2
$$

The selected disparity is:

$$
d^*(x,y)=\arg\min_d C(x,y,d)
$$

Once disparity is known, depth can be estimated with:

$$
Z(x,y)=\frac{fB}{d^*(x,y)}
$$

### Task 5

Run the synthetic stereo experiment. Compare the estimated disparity with the ground-truth disparity. Where does block matching make mistakes?

In [ ]:
def box_smooth(img, rounds=4):
    """Smooth an image using repeated 5-neighbor averaging."""
    out = img.astype(np.float32)
    for _ in range(rounds):
        out = (out + np.roll(out, 1, axis=0) + np.roll(out, -1, axis=0) +
               np.roll(out, 1, axis=1) + np.roll(out, -1, axis=1)) / 5.0
    return out


def make_synthetic_left_image(h=72, w=112):
    """Create a synthetic textured image with simple objects."""
    img = box_smooth(rng.random((h, w)), rounds=8)
    yy, xx = np.mgrid[0:h, 0:w]
    img[(yy > 18) & (yy < 48) & (xx > 18) & (xx < 46)] += 0.35
    img[((yy - 38) ** 2 + (xx - 78) ** 2) < 15 ** 2] += 0.45
    img[(yy > 52) & (xx > 8) & (xx < 98)] -= 0.25
    return normalize01(img)


def make_disparity_map(h=72, w=112):
    """Create a piecewise disparity map with depth discontinuities."""
    yy, xx = np.mgrid[0:h, 0:w]
    disp = np.full((h, w), 5.0, dtype=np.float32)
    disp[(yy > 18) & (yy < 48) & (xx > 18) & (xx < 46)] = 13.0
    disp[((yy - 38) ** 2 + (xx - 78) ** 2) < 15 ** 2] = 18.0
    disp[(yy > 52) & (xx > 8) & (xx < 98)] = 9.0
    return disp


def warp_left_to_right(left, disparity):
    """Create a right image from a left image using x_right = x_left - disparity."""
    h, w = left.shape
    right = np.full_like(left, 0.5)
    counts = np.zeros_like(left)
    for y in range(h):
        for x in range(w):
            xr = int(round(x - disparity[y, x]))
            if 0 <= xr < w:
                right[y, xr] += left[y, x]
                counts[y, xr] += 1
    mask = counts > 0
    right[mask] = right[mask] / counts[mask]
    right = box_smooth(right, rounds=1)
    return normalize01(right)

left_img = make_synthetic_left_image()
disp_gt = make_disparity_map(*left_img.shape)
right_img = warp_left_to_right(left_img, disp_gt)
right_img = normalize01(0.90 * right_img + 0.05 + rng.normal(0, 0.015, right_img.shape))

show_grid(
    [left_img, right_img, disp_gt],
    ["Left image", "Right image", "Ground-truth disparity"],
    cmap="gray",
    figsize=(14, 4)
)

In [ ]:
def block_match(left, right, max_disp=22, window=7, method="ssd"):
    """Estimate disparity with simple block matching on rectified images."""
    assert method in {"ssd", "sad", "ncc"}
    h, w = left.shape
    r = window // 2
    left_p = np.pad(left, r, mode="edge")
    right_p = np.pad(right, r, mode="edge")
    disp = np.zeros((h, w), dtype=np.float32)
    best_score = np.full((h, w), np.inf, dtype=np.float32)
    
    for y in range(h):
        for x in range(w):
            patch_l = left_p[y:y + window, x:x + window]
            for d in range(max_disp + 1):
                xr = x - d
                if xr < 0:
                    continue
                patch_r = right_p[y:y + window, xr:xr + window]
                if method == "ssd":
                    score = np.sum((patch_l - patch_r) ** 2)
                elif method == "sad":
                    score = np.sum(np.abs(patch_l - patch_r))
                else:
                    score = -ncc(patch_l.ravel(), patch_r.ravel())
                if score < best_score[y, x]:
                    best_score[y, x] = score
                    disp[y, x] = d
    return disp

estimated_disp = block_match(left_img, right_img, max_disp=22, window=7, method="ssd")
valid = disp_gt > 0
mae = np.mean(np.abs(estimated_disp[valid] - disp_gt[valid]))
print("Mean absolute disparity error:", mae)

show_grid(
    [disp_gt, estimated_disp, np.abs(estimated_disp - disp_gt)],
    ["Ground truth", "Estimated disparity", "Absolute error"],
    cmap="magma",
    figsize=(14, 4)
)

## 6. Window size trade-off

### Concept

Window size is a key design choice in block matching.

A smaller window usually gives:

- More detail near object boundaries.
- More noise in weakly textured regions.

A larger window usually gives:

- Smoother disparity maps.
- Less noise in flat regions.
- More errors near depth discontinuities because the window mixes foreground and background.

### Task 6

Compare `window=3`, `window=7`, and `window=15`. Which one gives the best result for this scene? Which one preserves boundaries best?

In [ ]:
window_sizes = [3, 7, 15]
results = []
for win in window_sizes:
    disp_est = block_match(left_img, right_img, max_disp=22, window=win, method="ssd")
    err = np.mean(np.abs(disp_est[valid] - disp_gt[valid]))
    results.append((win, disp_est, err))
    print(f"Window size {win:2d}: mean absolute error = {err:.3f}")

show_grid(
    [r[1] for r in results],
    [f"window={r[0]}, MAE={r[2]:.2f}" for r in results],
    cmap="magma",
    figsize=(14, 4)
)

## 7. Common failure cases

### Concept

Stereo block matching can fail when local patches are ambiguous or inconsistent across the two images.

Important failure cases include:

- **Textureless regions:** Many patches look almost the same.
- **Repeated patterns:** Multiple candidate matches have similar cost.
- **Specularities:** Appearance changes between views.
- **Occlusion:** A point visible in one camera may be hidden in the other.
- **Depth boundaries:** A window may contain pixels from multiple depths.

### Task 7

The next cell creates a repeated-pattern image. Try changing the stripe width. Explain why repeated patterns create multiple plausible disparities.

In [ ]:
h, w = 72, 112
yy, xx = np.mgrid[0:h, 0:w]
stripe_width = 8
left_repeated = ((xx // stripe_width) % 2).astype(np.float32)
left_repeated = 0.25 + 0.55 * left_repeated
left_repeated += 0.03 * rng.normal(size=left_repeated.shape)
left_repeated = normalize01(left_repeated)

disp_repeated = np.full((h, w), 12.0, dtype=np.float32)
right_repeated = warp_left_to_right(left_repeated, disp_repeated)

estimated_repeated = block_match(left_repeated, right_repeated, max_disp=24, window=7, method="ssd")

show_grid(
    [left_repeated, right_repeated, estimated_repeated],
    ["Left repeated pattern", "Right repeated pattern", "Estimated disparity"],
    cmap="gray",
    figsize=(14, 4)
)

print("True disparity:", 12)
print("Estimated disparity values and counts:")
values, counts = np.unique(estimated_repeated.astype(int), return_counts=True)
for value, count in zip(values, counts):
    if count > 100:
        print(f"d={value:2d}: {count} pixels")

## 8. Smoothness and energy minimization

### Concept

Block matching treats each pixel independently. This often produces noisy disparity maps. A common improvement is to optimize an energy with two terms:

$$
E(d)=\sum_p C(p,d_p)+\lambda\sum_{(p,q)\in\mathcal{N}}\rho(d_p-d_q)
$$

The data term $C(p,d_p)$ measures match quality. The smoothness term encourages neighboring pixels to have similar disparities.

Two common smoothness penalties are:

$$
\rho(a)=\mathbb{1}[a\ne 0]
$$

and

$$
\rho(a)=|a|
$$

For one scanline, we can minimize a simplified energy with dynamic programming:

$$
D_x(d)=C_x(d)+\min_{d'}\left(D_{x-1}(d')+\lambda |d-d'|\right)
$$

### Task 8

Run the dynamic programming example. Increase `lambda_smooth`. What happens to noise? What happens near sharp depth changes?

In [ ]:
def build_cost_volume_1d(left_row, right_row, max_disp=22, window=7):
    """Build a simple SSD cost volume for one scanline."""
    w = len(left_row)
    r = window // 2
    left_p = np.pad(left_row, r, mode="edge")
    right_p = np.pad(right_row, r, mode="edge")
    cost = np.full((w, max_disp + 1), np.inf, dtype=np.float32)
    for x in range(w):
        patch_l = left_p[x:x + window]
        for d in range(max_disp + 1):
            xr = x - d
            if xr < 0:
                continue
            patch_r = right_p[xr:xr + window]
            cost[x, d] = np.sum((patch_l - patch_r) ** 2)
    return cost


def dp_smooth_disparity(cost, lambda_smooth=0.8):
    """Find the best disparity sequence for one scanline using dynamic programming."""
    w, nd = cost.shape
    disparities = np.arange(nd)
    dp = np.zeros_like(cost)
    back = np.zeros((w, nd), dtype=np.int32)
    dp[0] = cost[0]
    
    for x in range(1, w):
        transition = dp[x - 1][None, :] + lambda_smooth * np.abs(disparities[:, None] - disparities[None, :])
        best_prev = np.argmin(transition, axis=1)
        dp[x] = cost[x] + transition[np.arange(nd), best_prev]
        back[x] = best_prev
    
    path = np.zeros(w, dtype=np.int32)
    path[-1] = np.argmin(dp[-1])
    for x in range(w - 2, -1, -1):
        path[x] = back[x + 1, path[x + 1]]
    return path

scanline_y = 38
cost_1d = build_cost_volume_1d(left_img[scanline_y], right_img[scanline_y], max_disp=22, window=7)
raw_1d = np.argmin(cost_1d, axis=1)
smoothed_1d = dp_smooth_disparity(cost_1d, lambda_smooth=0.8)
truth_1d = disp_gt[scanline_y]

plt.figure(figsize=(12, 4))
plt.plot(truth_1d, label="Ground truth")
plt.plot(raw_1d, label="Raw argmin")
plt.plot(smoothed_1d, label="DP with smoothness")
plt.xlabel("x position on one scanline")
plt.ylabel("disparity")
plt.title("Smoothness can reduce noise but may blur jumps")
plt.legend()
plt.grid(True)
plt.show()

## 9. Structured light

### Concept

Stereo matching is difficult because natural image appearance may not provide unique correspondences. Structured light makes matching easier by projecting known patterns onto the scene.

A projector can be treated like an inverse camera: instead of sensing rays, it emits coded rays. If each projector column has a unique temporal code, the camera can decode which projector column illuminated a surface point.

A simple binary code for a projector coordinate $x$ is:

$$
x = \sum_{b=0}^{n-1} 2^b c_b
$$

where each bit $c_b$ is observed from one projected pattern.

### Task 9

Run the binary structured-light example. Increase `nbits`. How does the number of uniquely encodable columns change?

In [ ]:
def make_binary_code_patterns(width=64, height=32, nbits=6):
    """Create binary stripe patterns that encode the x-coordinate."""
    x = np.arange(width)
    patterns = []
    for b in range(nbits):
        bit = ((x >> b) & 1).astype(np.float32)
        patterns.append(np.tile(bit[None, :], (height, 1)))
    return patterns

width, height, nbits = 64, 32, 6
patterns = make_binary_code_patterns(width=width, height=height, nbits=nbits)

# Decode by reading the bits at every image column.
decoded_x = np.zeros((height, width), dtype=np.int32)
for b, pattern in enumerate(patterns):
    decoded_x += (pattern > 0.5).astype(np.int32) * (2 ** b)

show_grid(
    patterns[:4] + [decoded_x],
    ["bit 0", "bit 1", "bit 2", "bit 3", "decoded x-coordinate"],
    cmap="gray",
    figsize=(15, 3)
)

print("Number of code bits:", nbits)
print("Maximum number of unique columns:", 2 ** nbits)
print("Decoded values on the first row:")
print(decoded_x[0, :16], "...")

## 10. Final challenge tasks

Use the code above to complete these open-ended tasks.

### Challenge A: Design a harder stereo pair

Modify `make_synthetic_left_image` and `make_disparity_map` to create a scene with:

- One textureless object.
- One repeated-pattern object.
- One thin object near a depth boundary.

Run block matching and describe which areas fail.

### Challenge B: Compare matching costs

Run `block_match` with `method="ssd"`, `method="sad"`, and `method="ncc"`.

Then modify the right image with:

$$
I_R' = aI_R + b
$$

Try several values of $a$ and $b$. Which matching cost is most robust?

### Challenge C: Improve the algorithm

Pick one improvement and implement it:

1. Left-right consistency checking.
2. Median filtering of the disparity map.
3. A confidence map based on the gap between the best and second-best matching costs.
4. A smoothness method applied to every scanline.

### Reflection questions

1. Why is rectification so important for stereo matching?
2. Why does a larger window help textureless regions but hurt depth boundaries?
3. Why is disparity inversely proportional to depth?
4. Why can structured light solve some correspondence problems that passive stereo cannot?

## Key takeaways

- Stereo depth depends on finding correct correspondences between two images.
- Rectification turns correspondence search into a 1D scanline search.
- Disparity and depth are related by $Z = \frac{fB}{d}$.
- Local matching costs such as SAD, SSD, and NCC have different robustness properties.
- Window size controls a trade-off between noise and boundary precision.
- Smoothness improves noisy estimates but can oversmooth real depth discontinuities.
- Structured light makes correspondence easier by adding known visual codes to the scene.